In [0]:
#Notebook 05: Silver to Gold
#Final dimensional model
#Build order:
# 1. dim_location
# 2. dim_date
# 3. dim_track
# 4. dim_playlist
# 5. bridge_playlist_track
# 6. dim_employee  (SCD Type 2)
# 7. dim_customer  (SCD Type 2)
# 8. fact_sales
# 9. fact_sales_customer_agg

#For testing
dbutils.widgets.text("catalog_name",  "chinook_catalog")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema",   "gold")

#Fetching values from job parameters
catalog = dbutils.widgets.get("catalog_name")
silver  = dbutils.widgets.get("silver_schema")
gold    = dbutils.widgets.get("gold_schema")

silver_db = f"`{catalog}`.`{silver}`"
gold_db   = f"`{catalog}`.`{gold}`"

print(f"Catalog : {catalog}")
print(f"Silver  : {silver_db}")
print(f"Gold    : {gold_db}")

#Importign required libraries
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, trim, lower, upper, concat_ws,
    monotonically_increasing_id, current_date,
    dayofmonth, month, date_format, quarter,
    year, weekofyear, coalesce, expr,
    row_number, sum as spark_sum, countDistinct
)
from pyspark.sql.types import DateType, IntegerType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

#Setup schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{gold}`")
print(f"Schema `{catalog}`.`{gold}` ready.\n")

today = current_date()

#Helper window for row_number-based surrogate keys to avoid monotonically_increasing_id overflow when cast to Int
w = Window.orderBy(monotonically_increasing_id())

Catalog : chinook_catalog
Silver  : `chinook_catalog`.`silver`
Gold    : `chinook_catalog`.`gold`
Schema `chinook_catalog`.`gold` ready.



In [0]:
#DIM_LOCATION: role-playing dimension
#Sources: Customer address, Employee address, Invoice billing address

print("Building dim_location...")

cust_addr = spark.table(f"{silver_db}.Customer").select(
    trim(col("Address")).alias("address"),
    trim(col("City")).alias("city"),
    trim(coalesce(col("State"), lit(""))).alias("state"),
    upper(trim(col("Country"))).alias("country"),
    trim(coalesce(col("PostalCode"), lit(""))).alias("postal_code")
)

emp_addr = spark.table(f"{silver_db}.Employee").select(
    trim(col("Address")).alias("address"),
    trim(col("City")).alias("city"),
    trim(coalesce(col("State"), lit(""))).alias("state"),
    upper(trim(col("Country"))).alias("country"),
    trim(coalesce(col("PostalCode"), lit(""))).alias("postal_code")
)

billing_addr = spark.table(f"{silver_db}.Invoice").select(
    trim(coalesce(col("BillingAddress"), lit(""))).alias("address"),
    trim(coalesce(col("BillingCity"), lit(""))).alias("city"),
    trim(coalesce(col("BillingState"), lit(""))).alias("state"),
    upper(trim(coalesce(col("BillingCountry"), lit("")))).alias("country"),
    trim(coalesce(col("BillingPostalCode"), lit(""))).alias("postal_code")
)

#Union all three sources and deduplicate
all_locations = cust_addr.union(emp_addr).union(billing_addr).distinct()

dim_location = all_locations \
    .withColumn("location_key", row_number().over(w).cast(IntegerType())) \
    .select("location_key", "address", "city", "state", "country", "postal_code")

dim_location.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"`{catalog}`.`{gold}`.dim_location")

print(f"dim_location: {dim_location.count()} rows\n")

Building dim_location...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_location: 71 rows



In [0]:
#DIM_DATE
#Full calendar from 2000-01-01 to 2030-12-31

print("Building dim_date...")

dim_date = spark.sql("""
    SELECT explode(sequence(
        to_date('2000-01-01'),
        to_date('2030-12-31'),
        interval 1 day
    )) AS full_date
""").select(
    expr("cast(date_format(full_date,'yyyyMMdd') as INT)").alias("date_key"),
    col("full_date"),
    dayofmonth("full_date").alias("day"),
    month("full_date").cast("string").alias("month"),
    date_format("full_date", "MMMM").alias("month_name"),
    quarter("full_date").alias("quarter"),
    year("full_date").alias("year"),
    weekofyear("full_date").alias("week_of_year")
)

dim_date.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"`{catalog}`.`{gold}`.dim_date")

print(f"dim_date: {dim_date.count()} rows\n")

Building dim_date...
dim_date: 11323 rows



In [0]:
#DIM_TRACK: flattened
#Joins Track, Album, Artist, Genre and MediaType into one dimension

print("Building dim_track...")

track     = spark.table(f"{silver_db}.Track")
album     = spark.table(f"{silver_db}.Album")
artist    = spark.table(f"{silver_db}.Artist")
genre     = spark.table(f"{silver_db}.Genre")
mediatype = spark.table(f"{silver_db}.MediaType")

dim_track = track \
    .join(album,     track.AlbumId     == album.AlbumId,       "left") \
    .join(artist,    album.ArtistId    == artist.ArtistId,      "left") \
    .join(genre,     track.GenreId     == genre.GenreId,        "left") \
    .join(mediatype, track.MediaTypeId == mediatype.MediaTypeId,"left") \
    .select(
        row_number().over(w).cast(IntegerType()).alias("track_key"),
        track.TrackId.cast(IntegerType()).alias("track_id"),
        trim(track.Name).alias("track_name"),
        track.AlbumId.cast(IntegerType()).alias("album_id"),
        trim(album.Title).alias("album_title"),
        track.MediaTypeId.cast(IntegerType()).alias("media_type_id"),
        trim(mediatype.Name).alias("media_type_name"),
        track.GenreId.cast(IntegerType()).alias("genre_id"),
        trim(genre.Name).alias("genre_name"),
        album.ArtistId.cast(IntegerType()).alias("artist_id"),
        trim(artist.Name).alias("artist_name"),
        trim(coalesce(track.Composer, lit(""))).alias("composer"),
        track.Milliseconds.cast(IntegerType()).alias("milliseconds"),
        track.Bytes.cast(IntegerType()).alias("bytes")
    )

dim_track.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"`{catalog}`.`{gold}`.dim_track")

print(f"dim_track: {dim_track.count()} rows\n")

Building dim_track...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_track: 3503 rows



In [0]:
#DIM_PLAYLIST

print("Building dim_playlist...")

dim_playlist = spark.table(f"{silver_db}.Playlist").select(
    row_number().over(w).cast(IntegerType()).alias("playlist_key"),
    col("PlaylistId").cast(IntegerType()).alias("playlist_id"),
    trim(col("Name")).alias("playlist_name")
)

dim_playlist.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"`{catalog}`.`{gold}`.dim_playlist")

print(f"dim_playlist: {dim_playlist.count()} rows\n")

Building dim_playlist...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_playlist: 18 rows



In [0]:
#BRIDGE_PLAYLIST_TRACK
#Bride table to handle many-to-many relationship between playlists and tracks

print("Building bridge_playlist_track...")

pt_src    = spark.table(f"{silver_db}.PlaylistTrack")
pl_lkp    = spark.table(f"{gold_db}.dim_playlist").select("playlist_key", "playlist_id")
track_lkp = spark.table(f"{gold_db}.dim_track").select("track_key", "track_id")

bridge = pt_src \
    .join(pl_lkp, pt_src.PlaylistId == pl_lkp.playlist_id, "left") \
    .join(track_lkp, pt_src.TrackId == track_lkp.track_id, "left") \
    .select(
        track_lkp.track_key.cast(IntegerType()),
        pl_lkp.playlist_key.cast(IntegerType())
    )

bridge.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"`{catalog}`.`{gold}`.bridge_playlist_track")

print(f"bridge_playlist_track: {bridge.count()} rows\n")

Building bridge_playlist_track...
bridge_playlist_track: 8715 rows



In [0]:
#DIM_EMPLOYEE: SCD Type 2
#Using row_number() instead of monotonically_increasing_id() because of conversion to int issue

print("Building dim_employee...")

emp_src = spark.table(f"{silver_db}.Employee")
loc_lkp = spark.table(f"{gold_db}.dim_location") \
               .select("location_key", "address", "city", "state", "country", "postal_code")

incoming_emp = emp_src \
    .join(
        loc_lkp,
        (trim(emp_src.Address) == loc_lkp.address) &
        (trim(emp_src.City) == loc_lkp.city) &
        (upper(trim(coalesce(emp_src.Country, lit("")))) == loc_lkp.country) &
        (trim(coalesce(emp_src.State, lit(""))) == loc_lkp.state) &
        (trim(coalesce(emp_src.PostalCode, lit(""))) == loc_lkp.postal_code),
        "left"
    ) \
    .select(
        emp_src.EmployeeId.cast(IntegerType()).alias("employee_id"),
        trim(emp_src.FirstName).alias("first_name"),     
        trim(emp_src.LastName).alias("last_name"),         
        concat_ws(" ", trim(emp_src.FirstName), trim(emp_src.LastName)).alias("full_name"),
        trim(emp_src.Title).alias("title"),                
        emp_src.ReportsTo.cast("string").alias("reports_to"),
        emp_src.BirthDate.cast("date").alias("birth_date"),
        emp_src.HireDate.cast("date").alias("hire_date"),
        trim(emp_src.Phone).alias("phone"),                
        trim(emp_src.Fax).alias("fax"),                    
        lower(trim(emp_src.Email)).alias("email"),         
        loc_lkp.location_key.cast(IntegerType()).alias("address_key")  
    )

emp_exists = spark.catalog.tableExists(f"`{catalog}`.`{gold}`.dim_employee")

if not emp_exists:
    #First Load
    dim_emp_new = incoming_emp \
        .withColumn("employee_key", row_number().over(w).cast(IntegerType())) \
        .withColumn("effective_start_date", today) \
        .withColumn("effective_end_date", lit(None).cast(DateType())) \
        .withColumn("is_current", lit(True))

    dim_emp_new.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"`{catalog}`.`{gold}`.dim_employee")
    print(f"dim_employee first load: {dim_emp_new.count()} rows\n")

else:
    #Subsequent Loads
    existing_emp = spark.table(f"{gold_db}.dim_employee").filter("is_current = true")

    changed_emp = incoming_emp.alias("src").join(
        existing_emp.alias("tgt"), "employee_id"
    ).filter(
        ~col("src.first_name").eqNullSafe(col("tgt.first_name"))   |
        ~col("src.last_name").eqNullSafe(col("tgt.last_name"))     |
        ~col("src.title").eqNullSafe(col("tgt.title"))             |
        ~col("src.email").eqNullSafe(col("tgt.email"))             |
        ~col("src.phone").eqNullSafe(col("tgt.phone"))             |
        ~col("src.fax").eqNullSafe(col("tgt.fax"))                 |
        ~col("src.address_key").eqNullSafe(col("tgt.address_key"))
    ).select("src.*")

    #For brand new employees who are not in Gold at all
    new_emp = incoming_emp.alias("src").join(
        existing_emp.alias("tgt"), "employee_id", "left_anti"
    )

    to_insert_emp = changed_emp.union(new_emp)

    if to_insert_emp.count() > 0:
        changed_emp_ids = [r["employee_id"] for r in changed_emp.select("employee_id").collect()]

        if changed_emp_ids:
            #Closing old records for changed or updated employee records
            dt_emp = DeltaTable.forName(spark, f"`{catalog}`.`{gold}`.dim_employee")
            dt_emp.update(
                condition=f"employee_id IN ({','.join(str(i) for i in changed_emp_ids)}) AND is_current = true",
                set={
                    "effective_end_date": "current_date() - interval 1 day",
                    "is_current": "false"
                }
            )
            print(f"Closed {len(changed_emp_ids)} existing employee records")

        #Inserting new records for changed/Updated/New employees
        new_emp_rows = to_insert_emp \
            .withColumn("employee_key", row_number().over(w).cast(IntegerType())) \
            .withColumn("effective_start_date", today) \
            .withColumn("effective_end_date", lit(None).cast(DateType())) \
            .withColumn("is_current", lit(True))

        new_emp_rows.write.format("delta").mode("append") \
            .saveAsTable(f"`{catalog}`.`{gold}`.dim_employee")
        print(f"Inserted {to_insert_emp.count()} new/changed employee rows")
    else:
        print("dim_employee: no changes detected")

print(f"dim_employee total: {spark.table(f'{gold_db}.dim_employee').count()} rows\n")

Building dim_employee...
Closed 8 existing employee records


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Inserted 0 new/changed employee rows
dim_employee total: 16 rows



In [0]:
#DIM_CUSTOMER: SCD Type 2
#Using row_number() instead of monotonically_increasing_id() because of conversion to int issue

print("Building dim_customer...")

cust_src = spark.table(f"{silver_db}.Customer")
loc_lkp  = spark.table(f"{gold_db}.dim_location") \
               .select("location_key", "address", "city", "state", "country", "postal_code")

incoming_cust = cust_src \
    .join(
        loc_lkp,
        (trim(cust_src.Address) == loc_lkp.address) &
        (trim(cust_src.City) == loc_lkp.city) &
        (upper(trim(coalesce(cust_src.Country, lit("")))) == loc_lkp.country) &
        (trim(coalesce(cust_src.State, lit(""))) == loc_lkp.state) &
        (trim(coalesce(cust_src.PostalCode, lit(""))) == loc_lkp.postal_code),
        "left"
    ) \
    .select(
        cust_src.CustomerId.cast(IntegerType()).alias("customer_id"),
        trim(cust_src.FirstName).alias("first_name"),      
        trim(cust_src.LastName).alias("last_name"),         
        concat_ws(" ", trim(cust_src.FirstName), trim(cust_src.LastName)).alias("full_name"),
        trim(cust_src.Company).alias("company"),           
        lower(trim(cust_src.Email)).alias("email"),         
        trim(cust_src.Phone).alias("phone"),                
        trim(cust_src.Fax).alias("fax"),                    
        cust_src.SupportRepId.cast(IntegerType()).alias("support_rep_id"),  
        loc_lkp.location_key.cast(IntegerType()).alias("address_key")       
    )

cust_exists = spark.catalog.tableExists(f"`{catalog}`.`{gold}`.dim_customer")

if not cust_exists:
    #First Load
    dim_cust_new = incoming_cust \
        .withColumn("customer_key", row_number().over(w).cast(IntegerType())) \
        .withColumn("effective_start_date", today) \
        .withColumn("effective_end_date", lit(None).cast(DateType())) \
        .withColumn("is_current", lit(True))

    dim_cust_new.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"`{catalog}`.`{gold}`.dim_customer")
    print(f"dim_customer first load: {dim_cust_new.count()} rows\n")

else:
    #Subsequent Loads
    existing_cust = spark.table(f"{gold_db}.dim_customer").filter("is_current = true")

    changed_cust = incoming_cust.alias("src").join(
        existing_cust.alias("tgt"), "customer_id"
    ).filter(
        ~col("src.first_name").eqNullSafe(col("tgt.first_name"))         |
        ~col("src.last_name").eqNullSafe(col("tgt.last_name"))           |
        ~col("src.email").eqNullSafe(col("tgt.email"))                   |
        ~col("src.company").eqNullSafe(col("tgt.company"))               |  
        ~col("src.phone").eqNullSafe(col("tgt.phone"))                   |  
        ~col("src.fax").eqNullSafe(col("tgt.fax"))                       |  
        ~col("src.support_rep_id").eqNullSafe(col("tgt.support_rep_id")) |
        ~col("src.address_key").eqNullSafe(col("tgt.address_key"))
    ).select("src.*")

    #For brand new employees who are not in Gold at all
    new_cust = incoming_cust.alias("src").join(
        existing_cust.alias("tgt"), "customer_id", "left_anti"
    )

    to_insert_cust = changed_cust.union(new_cust)

    if to_insert_cust.count() > 0:
        changed_cust_ids = [r["customer_id"] for r in changed_cust.select("customer_id").collect()]

        if changed_cust_ids:
            #Closing old records for changed or updated employee records
            dt_cust = DeltaTable.forName(spark, f"`{catalog}`.`{gold}`.dim_customer")
            dt_cust.update(
                condition=f"customer_id IN ({','.join(str(i) for i in changed_cust_ids)}) AND is_current = true",
                set={
                    "effective_end_date": "current_date() - interval 1 day",
                    "is_current":"false"
                }
            )
            print(f"Closed {len(changed_cust_ids)} existing customer records")

        #Inserting new records for changed/Updated/New employees
        new_cust_rows = to_insert_cust \
            .withColumn("customer_key",         row_number().over(w).cast(IntegerType())) \
            .withColumn("effective_start_date", today) \
            .withColumn("effective_end_date",   lit(None).cast(DateType())) \
            .withColumn("is_current",           lit(True))

        new_cust_rows.write.format("delta").mode("append") \
            .saveAsTable(f"`{catalog}`.`{gold}`.dim_customer")
        print(f"Inserted {to_insert_cust.count()} new/changed customer rows")
    else:
        print("dim_customer: no changes detected")

print(f"dim_customer total: {spark.table(f'{gold_db}.dim_customer').count()} rows\n")

Building dim_customer...
Closed 28 existing customer records


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Inserted 0 new/changed customer rows
dim_customer total: 89 rows



In [0]:
#FACT_SALES
#Grain: one row per invoice line item

print("Building fact_sales...")

inv  = spark.table(f"{silver_db}.Invoice")
invl = spark.table(f"{silver_db}.InvoiceLine")

#Dimension lookups (only current records for SCD2 dims)
dim_cust_lkp  = spark.table(f"{gold_db}.dim_customer") \
                     .filter("is_current = true") \
                     .select("customer_key", "customer_id")

dim_emp_lkp   = spark.table(f"{gold_db}.dim_employee") \
                     .filter("is_current = true") \
                     .select("employee_key", "employee_id")

dim_track_lkp = spark.table(f"{gold_db}.dim_track") \
                     .select("track_key", "track_id")

dim_date_lkp  = spark.table(f"{gold_db}.dim_date") \
                     .select("date_key")

#Billing address lookup from dim_location
billing_lkp   = spark.table(f"{gold_db}.dim_location") \
                     .select(
                         col("location_key").alias("billing_address_key"),
                         col("address").alias("b_address"),
                         col("city").alias("b_city"),
                         col("state").alias("b_state"),
                         col("country").alias("b_country"),
                         col("postal_code").alias("b_postal_code")
                     )

# Support rep lookup via Customer
cust_rep_lkp  = spark.table(f"{silver_db}.Customer") \
                     .select(
                         col("CustomerId").alias("cust_id"),
                         col("SupportRepId").alias("support_rep_id")
                     )

fact_sales_raw = invl.alias("il") \
    .join(inv.alias("i"),
          col("il.InvoiceId") == col("i.InvoiceId"), "inner") \
    .join(dim_cust_lkp.alias("dc"),
          col("i.CustomerId") == col("dc.customer_id"), "left") \
    .join(cust_rep_lkp.alias("cr"),
          col("i.CustomerId") == col("cr.cust_id"), "left") \
    .join(dim_emp_lkp.alias("de"),
          col("cr.support_rep_id") == col("de.employee_id"), "left") \
    .join(dim_track_lkp.alias("dt"),
          col("il.TrackId") == col("dt.track_id"), "left") \
    .join(dim_date_lkp.alias("dd"),
          expr("cast(date_format(i.InvoiceDate,'yyyyMMdd') as INT)") == col("dd.date_key"), "left") \
    .join(
          billing_lkp.alias("bl"),
          (trim(col("i.BillingAddress")) == col("bl.b_address")) &
          (trim(col("i.BillingCity")) == col("bl.b_city")) &
          (upper(trim(coalesce(col("i.BillingCountry"), lit("")))) == col("bl.b_country")) &
          (trim(coalesce(col("i.BillingState"), lit(""))) == col("bl.b_state")) &
          (trim(coalesce(col("i.BillingPostalCode"), lit(""))) == col("bl.b_postal_code")),
          "left") \
    .select(
        col("i.InvoiceId").cast(IntegerType()).alias("invoice_id"),
        col("il.InvoiceLineId").cast(IntegerType()).alias("invoice_line_id"),
        col("dd.date_key").cast(IntegerType()).alias("invoice_date_key"),
        col("dc.customer_key").cast(IntegerType()).alias("customer_key"),
        col("de.employee_key").cast(IntegerType()).alias("employee_key"),
        col("dt.track_key").cast(IntegerType()).alias("track_key"),
        col("bl.billing_address_key").cast(IntegerType()).alias("billing_address_key"),
        col("il.Quantity").cast(IntegerType()).alias("quantity"),
        col("il.UnitPrice").alias("unit_price"),
        (col("il.Quantity") * col("il.UnitPrice")).alias("sales_amount")
    )

#Adding sales_key using row_number 
fact_sales = fact_sales_raw \
    .withColumn("sales_key", row_number().over(w).cast(IntegerType())) \
    .select(
        "sales_key", "invoice_id", "invoice_line_id", "invoice_date_key",
        "customer_key", "employee_key", "track_key", "billing_address_key",
        "quantity", "unit_price", "sales_amount"
    )

fact_sales.write.format("delta") \
    .mode("append") \
#   .option("overwriteSchema", "true") \
    .saveAsTable(f"`{catalog}`.`{gold}`.fact_sales")

print(f"fact_sales: {fact_sales.count()} rows\n")

Building fact_sales...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


fact_sales: 2240 rows



In [0]:
#FACT_SALES_CUSTOMER_AGG
#Source: gold.fact_sales
#Grain: one row per customer per invoice date

print("Building fact_sales_customer_agg...")

fs = spark.table(f"{gold_db}.fact_sales")

agg = fs.groupBy("customer_key", "invoice_date_key") \
    .agg(
        countDistinct("invoice_id").alias("total_invoices"),
        spark_sum("quantity").alias("total_quantity"),
        spark_sum("sales_amount").alias("total_sales_amount")
    ) \
    .withColumn(
        "avg_invoice_value",
        col("total_sales_amount") / col("total_invoices")
    )

agg.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"`{catalog}`.`{gold}`.fact_sales_customer_agg")

print(f"fact_sales_customer_agg: {agg.count()} rows\n")

Building fact_sales_customer_agg...
fact_sales_customer_agg: 412 rows



In [0]:
#GOLD LAYER VALIDATION

print("============== GOLD LAYER VALIDATION ==============")
gold_tables = [
    "dim_location",
    "dim_date",
    "dim_track",
    "dim_playlist",
    "bridge_playlist_track",
    "dim_employee",
    "dim_customer",
    "fact_sales",
    "fact_sales_customer_agg"
]

for t in gold_tables:
    cnt = spark.table(f"`{catalog}`.`{gold}`.{t}").count()
    print(f"  {t:<35} {cnt:>8} rows")

#SCD2 sanity checks
for dim in ["dim_customer", "dim_employee"]:
    active   = spark.table(f"{gold_db}.{dim}").filter("is_current = true").count()
    inactive = spark.table(f"{gold_db}.{dim}").filter("is_current = false").count()
    print(f"\n{dim}")
    print(f"  active   (is_current=true) : {active}")
    print(f"  inactive (is_current=false): {inactive}")

print("\n===============================================")
print("Gold Layer Complete")
print("===============================================")

============== GOLD LAYER VALIDATION ==============
  dim_location                              71 rows
  dim_date                               11323 rows
  dim_track                               3503 rows
  dim_playlist                              18 rows
  bridge_playlist_track                   8715 rows
  dim_employee                              16 rows
  dim_customer                              89 rows
  fact_sales                              2240 rows
  fact_sales_customer_agg                  412 rows

dim_customer
  active   (is_current=true) : 61
  inactive (is_current=false): 28

dim_employee
  active   (is_current=true) : 8
  inactive (is_current=false): 8

Gold Layer Complete
